In [1]:
# Dependências principais
from __future__ import annotations

from typing import Dict, Iterable, Set, Tuple

import nltk

for resource in ("punkt",):
    try:
        nltk.data.find(f"tokenizers/{resource}")
    except LookupError:
        nltk.download(resource)

print("Recursos do NLTK prontos.")

Recursos do NLTK prontos.


In [2]:
# Funções de tokenização, cálculo de Jaccard e decisão de plágio
def tokenize(texto: str) -> Iterable[str]:
    return (
        token.lower()
        for token in nltk.word_tokenize(texto, language="portuguese")
        if token.isalpha()
    )

def vocabulario(texto: str) -> Set[str]:
    return set(tokenize(texto))

def similaridade_jaccard(doc_a: str, doc_b: str) -> Tuple[float, Set[str], Set[str]]:
    vocab_a = vocabulario(doc_a)
    vocab_b = vocabulario(doc_b)
    uniao = vocab_a | vocab_b
    intersecao = vocab_a & vocab_b
    indice = len(intersecao) / len(uniao) if uniao else 0.0
    return indice, vocab_a, vocab_b

def detectar_plagio(doc_referencia: str, doc_suspeito: str, limiar: float = 0.5) -> Dict[str, float | int | str]:
    indice, vocab_a, vocab_b = similaridade_jaccard(doc_referencia, doc_suspeito)
    resultado = "PLÁGIO" if indice > limiar else "OK"
    return {
        "indice": indice,
        "resultado": resultado,
        "limiar": limiar,
        "vocab_referencia": len(vocab_a),
        "vocab_suspeito": len(vocab_b),
        "tokens_comuns": len(vocab_a & vocab_b),
        "tamanho_uniao": len(vocab_a | vocab_b),
    }

def imprimir_relatorio(titulo: str, relatorio: Dict[str, float | int | str]) -> None:
    print(titulo)
    print(
        f"  Similaridade de Jaccard: {relatorio['indice']*100:.2f}% -> {relatorio['resultado']} ",
        f"(limiar {relatorio['limiar']*100:.0f}%)"
    )
    print(
        f"  Vocabulários: {relatorio['vocab_referencia']} vs {relatorio['vocab_suspeito']} ",
        f"| Interseção: {relatorio['tokens_comuns']} | União: {relatorio['tamanho_uniao']}"
    )
    print()

In [3]:
referencia = (
    "O pesquisador descreveu meticulosamente o bairro, suas vielas, o comércio "
    "popular e as mudanças sociais percebidas ao longo de décadas."
)
suspeito_plagio = referencia + " O mesmo relato foi copiado integralmente, sem citar a fonte."
suspeito_autoral = (
    "Uma crônica rural narra as colheitas, festas populares e memórias de infância "
    "vividas no interior nordestino."
)

relatorio_plagio = detectar_plagio(referencia, suspeito_plagio)
imprimir_relatorio("Caso suspeito", relatorio_plagio)

relatorio_autoral = detectar_plagio(referencia, suspeito_autoral)
imprimir_relatorio("Caso autoral", relatorio_autoral)

Caso suspeito
  Similaridade de Jaccard: 66.67% -> PLÁGIO  (limiar 50%)
  Vocabulários: 18 vs 27  | Interseção: 18 | União: 27

Caso autoral
  Similaridade de Jaccard: 9.68% -> OK  (limiar 50%)
  Vocabulários: 18 vs 16  | Interseção: 3 | União: 31

